# Mini Project: Complete ML Pipeline

End-to-end machine learning project: data preprocessing, model training, evaluation, and deployment.

## The ML Pipeline

1. **Data Collection** ← We're starting here
2. **Data Cleaning** ← Handle missing values, outliers
3. **Feature Engineering** ← Create new features
4. **Model Training** ← Train multiple models
5. **Model Evaluation** ← Cross-validation, metrics
6. **Hyperparameter Tuning** ← Optimise parameters
7. **Model Selection** ← Choose best model
8. **Model Deployment** ← Save and serve model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Data Collection

Load the breast cancer dataset. In real projects, this could be CSV files, databases, APIs, etc.

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Create DataFrame
df = pd.concat([X, y], axis=1)

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"\nFeatures: {len(data.feature_names)}")
print(f"Target classes: {data.target_names}")
print(f"\nClass distribution:")
print(df['target'].value_counts())
print(f"\n{df['target'].value_counts(normalize=True) * 100}")

# Basic info
print("\nData types and missing values:")
print(df.info())

# Summary statistics
print("\nSummary statistics:")
print(df.describe())

# Check for missing values
print(f"\nMissing values: {df.isnull().sum().sum()}")

# Target distribution
plt.figure(figsize=(8, 5))
df['target'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks([0, 1], data.target_names)
plt.grid(True, alpha=0.3)
plt.show()

## 2. Data Cleaning

Handle missing values, outliers, and data quality issues.

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Check for outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return ((data[column] < lower_bound) | (data[column] > upper_bound)).sum()

# Check outliers for all features
outlier_counts = {}
for col in X.columns:
    outlier_counts[col] = detect_outliers_iqr(df, col)

outlier_df = pd.DataFrame(list(outlier_counts.items()), columns=['Feature', 'Outliers'])
outlier_df = outlier_df.sort_values('Outliers', ascending=False)

print("\nOutlier counts by feature:")
print(outlier_df.head(10))

# Visualize outliers for top features
top_outlier_features = outlier_df.head(3)['Feature'].values

plt.figure(figsize=(15, 5))
for i, feature in enumerate(top_outlier_features):
    plt.subplot(1, 3, i+1)
    sns.boxplot(x=df['target'], y=df[feature])
    plt.title(f'{feature} by Class')
    plt.xticks([0, 1], data.target_names)

plt.tight_layout()
plt.show()

# Correlation analysis
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm', 
            square=True, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix')
plt.show()

# Highly correlated features
high_corr = correlation_matrix.where(
    np.triu(np.ones_like(correlation_matrix), k=1).astype(bool)
).stack().sort_values(ascending=False)

print("\nTop 10 highly correlated feature pairs:")
print(high_corr.head(10))

# For this dataset, we'll keep all features since it's a standard benchmark
# In practice, you might remove highly correlated features to reduce multicollinearity

## 3. Feature Engineering

Create new features and prepare data for modeling.

In [ ]:
# Split data first (before any preprocessing to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Feature engineering on training data only
def create_features(X):
    """Create new features from existing ones"""
    X_new = X.copy()
    
    # Area-based features
    area_features = [col for col in X.columns if 'area' in col.lower()]
    if len(area_features) > 1:
        X_new['area_mean'] = X[area_features].mean(axis=1)
        X_new['area_std'] = X[area_features].std(axis=1)
    
    # Perimeter-based features
    perimeter_features = [col for col in X.columns if 'perimeter' in col.lower()]
    if len(perimeter_features) > 1:
        X_new['perimeter_mean'] = X[perimeter_features].mean(axis=1)
        X_new['perimeter_std'] = X[perimeter_features].std(axis=1)
    
    # Texture-based features
    texture_features = [col for col in X.columns if 'texture' in col.lower()]
    if len(texture_features) > 1:
        X_new['texture_mean'] = X[texture_features].mean(axis=1)
        X_new['texture_std'] = X[texture_features].std(axis=1)
    
    # Compactness ratios
    if 'area_worst' in X.columns and 'perimeter_worst' in X.columns:
        X_new['compactness_ratio'] = X['perimeter_worst'] ** 2 / X['area_worst']
    
    return X_new

# Create features
X_train_fe = create_features(X_train)
X_test_fe = create_features(X_test)

print(f"\nOriginal features: {X_train.shape[1]}")
print(f"After feature engineering: {X_train_fe.shape[1]}")
print(f"New features added: {X_train_fe.shape[1] - X_train.shape[1]}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_fe)
X_test_scaled = scaler.transform(X_test_fe)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_fe.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_fe.columns)

print("\nFeature scaling completed.")
print(f"Training data mean: {X_train_scaled.mean().mean():.6f}")
print(f"Training data std: {X_train_scaled.std().mean():.6f}")

## 4. Model Training

Train multiple models and compare their baseline performance.

In [ ]:
# Define models to try
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True)
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    # Train on full training set
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Probabilities for ROC
    if hasattr(model, 'predict_proba'):
        y_prob_test = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob_test = model.decision_function(X_test_scaled)
    
    # Metrics
    results[name] = {
        'cv_accuracy': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'train_accuracy': accuracy_score(y_train, y_pred_train),
        'test_accuracy': accuracy_score(y_test, y_pred_test),
        'precision': precision_score(y_test, y_pred_test),
        'recall': recall_score(y_test, y_pred_test),
        'f1': f1_score(y_test, y_pred_test),
        'y_prob': y_prob_test,
        'model': model
    }
    
    print(f"CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    print(f"Train Accuracy: {results[name]['train_accuracy']:.3f}")
    print(f"Test Accuracy: {results[name]['test_accuracy']:.3f}")
    print(f"Precision: {results[name]['precision']:.3f}")
    print(f"Recall: {results[name]['recall']:.3f}")
    print(f"F1 Score: {results[name]['f1']:.3f}")

# Compare models
comparison_df = pd.DataFrame({
    name: {
        'CV Accuracy': f"{results[name]['cv_accuracy']:.3f} ± {results[name]['cv_std']:.3f}",
        'Test Accuracy': f"{results[name]['test_accuracy']:.3f}",
        'Precision': f"{results[name]['precision']:.3f}",
        'Recall': f"{results[name]['recall']:.3f}",
        'F1 Score': f"{results[name]['f1']:.3f}"
    } for name in models.keys()
}).T

print("\nModel Comparison:")
print(comparison_df)

# Visualize comparison
metrics = ['test_accuracy', 'precision', 'recall', 'f1']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

plt.figure(figsize=(15, 10))

for i, (metric, name) in enumerate(zip(metrics, metric_names)):
    plt.subplot(2, 2, i+1)
    values = [results[model][metric] for model in models.keys()]
    bars = plt.bar(models.keys(), values, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
    plt.title(f'Test {name}')
    plt.ylabel(name)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 5. Model Evaluation

Detailed evaluation with confusion matrices and ROC curves.

In [ ]:
# Confusion matrices and ROC curves
plt.figure(figsize=(16, 12))

for i, name in enumerate(models.keys()):
    model_results = results[name]
    model = model_results['model']
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Confusion Matrix
    plt.subplot(len(models), 3, i*3 + 1)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=data.target_names, yticklabels=data.target_names)
    plt.title(f'{name}\nConfusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Classification Report
    plt.subplot(len(models), 3, i*3 + 2)
    report = classification_report(y_test, y_pred, target_names=data.target_names, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    sns.heatmap(report_df.iloc[:-1, :-1], annot=True, cmap='RdYlGn', fmt='.3f')
    plt.title(f'{name}\nClassification Report')
    
    # ROC Curve
    plt.subplot(len(models), 3, i*3 + 3)
    if hasattr(model, 'predict_proba'):
        y_prob = model_results['y_prob']
    else:
        y_prob = model.decision_function(X_test_scaled)
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, color='darkorange', linewidth=2, 
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', linewidth=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{name}\nROC Curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Best model selection
best_model_name = max(results.keys(), key=lambda x: results[x]['f1'])
best_model = results[best_model_name]['model']

print(f"\nBest model based on F1 score: {best_model_name}")
print(f"F1 Score: {results[best_model_name]['f1']:.3f}")
print(f"Test Accuracy: {results[best_model_name]['test_accuracy']:.3f}")

# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train_fe.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x='importance', y='feature', data=feature_importance.head(15))
    plt.title(f'Feature Importance - {best_model_name}')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\nTop 10 important features:")
    print(feature_importance.head(10))

## 6. Hyperparameter Tuning

Optimise the best model's hyperparameters.

In [ ]:
# Hyperparameter grids for different models
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    },
    'Decision Tree': {
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'SVM': {
        'C': [0.1, 1.0, 10.0, 100.0],
        'kernel': ['linear', 'rbf', 'poly'],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
    }
}

# Tune the best model
print(f"Tuning hyperparameters for {best_model_name}...")

# Create base model
if best_model_name == 'Logistic Regression':
    base_model = LogisticRegression(random_state=42, max_iter=1000)
elif best_model_name == 'Decision Tree':
    base_model = DecisionTreeClassifier(random_state=42)
elif best_model_name == 'Random Forest':
    base_model = RandomForestClassifier(random_state=42)
else:  # SVM
    base_model = SVC(random_state=42, probability=True)

# Grid search
grid_search = GridSearchCV(
    base_model, 
    param_grids[best_model_name], 
    cv=5, 
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.3f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_scaled)

tuned_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_tuned),
    'precision': precision_score(y_test, y_pred_tuned),
    'recall': recall_score(y_test, y_pred_tuned),
    'f1': f1_score(y_test, y_pred_tuned)
}

print("\nTuned Model Performance:")
for metric, value in tuned_metrics.items():
    print(f"{metric.capitalize()}: {value:.3f}")

# Compare before and after tuning
comparison = pd.DataFrame({
    'Before Tuning': [
        results[best_model_name]['test_accuracy'],
        results[best_model_name]['precision'],
        results[best_model_name]['recall'],
        results[best_model_name]['f1']
    ],
    'After Tuning': [
        tuned_metrics['accuracy'],
        tuned_metrics['precision'],
        tuned_metrics['recall'],
        tuned_metrics['f1']
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

print("\nBefore vs After Tuning:")
print(comparison)

# Improvement
improvement = tuned_metrics['f1'] - results[best_model_name]['f1']
print(f"\nF1 Score improvement: {improvement:.3f} ({improvement*100:.1f}%)")

## 7. Model Selection & Deployment

Save the best model and create a simple prediction function.

In [ ]:
# Save the best model and preprocessing objects
model_artifacts = {
    'model': tuned_model,
    'scaler': scaler,
    'feature_names': list(X_train_fe.columns),
    'target_names': data.target_names,
    'model_name': best_model_name,
    'performance': tuned_metrics
}

# Save to disk
joblib.dump(model_artifacts, 'breast_cancer_model.pkl')
print("Model saved to 'breast_cancer_model.pkl'")

# Create prediction function
def predict_breast_cancer(features_dict):
    """
    Predict breast cancer diagnosis from features.
    
    Args:
        features_dict: Dictionary with feature names as keys
    
    Returns:
        dict: Prediction results
    """
    # Load model
    artifacts = joblib.load('breast_cancer_model.pkl')
    model = artifacts['model']
    scaler = artifacts['scaler']
    feature_names = artifacts['feature_names']
    target_names = artifacts['target_names']
    
    # Create feature vector
    features = pd.DataFrame([features_dict])
    
    # Add engineered features
    features = create_features(features)
    
    # Ensure correct feature order
    features = features[feature_names]
    
    # Scale features
    features_scaled = scaler.transform(features)
    
    # Make prediction
    prediction = model.predict(features_scaled)[0]
    probability = model.predict_proba(features_scaled)[0]
    
    return {
        'prediction': target_names[prediction],
        'probability_benign': probability[0],
        'probability_malignant': probability[1],
        'confidence': max(probability)
    }

# Test the prediction function
sample_features = X_test.iloc[0].to_dict()
result = predict_breast_cancer(sample_features)

print("\nPrediction Function Test:")
print(f"Predicted: {result['prediction']}")
print(f"Actual: {data.target_names[y_test.iloc[0]]}")
print(f"Confidence: {result['confidence']:.3f}")

# Model summary
print("\n=== MODEL DEPLOYMENT SUMMARY ===")
print(f"Model Type: {best_model_name}")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test F1 Score: {tuned_metrics['f1']:.3f}")
print(f"Test Accuracy: {tuned_metrics['accuracy']:.3f}")
print(f"Features Used: {len(X_train_fe.columns)}")
print("Model saved and ready for deployment!")

# Final visualization - model comparison with tuned model
plt.figure(figsize=(12, 6))

# All models + tuned
all_models = list(models.keys()) + [f'{best_model_name} (Tuned)']
all_f1_scores = [results[name]['f1'] for name in models.keys()] + [tuned_metrics['f1']]

bars = plt.bar(all_models, all_f1_scores, 
               color=['skyblue']*4 + ['darkgreen'])
plt.title('Model Performance Comparison (F1 Score)')
plt.ylabel('F1 Score')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Add value labels
for bar, score in zip(bars, all_f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{score:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Pipeline Summary

Congratulations! You've completed a full ML pipeline:

1. ✅ **Data Collection**: Loaded and explored breast cancer dataset
2. ✅ **Data Cleaning**: Checked for missing values, outliers, correlations
3. ✅ **Feature Engineering**: Created aggregate features, scaled data
4. ✅ **Model Training**: Trained 4 different models
5. ✅ **Model Evaluation**: Comprehensive evaluation with multiple metrics
6. ✅ **Hyperparameter Tuning**: Optimised the best model
7. ✅ **Model Selection**: Selected best performing model
8. ✅ **Model Deployment**: Saved model and created prediction function

## Key Takeaways

- **Systematic approach** matters more than fancy algorithms
- **Cross-validation** prevents overfitting to test set
- **Feature engineering** can significantly improve performance
- **Hyperparameter tuning** often gives meaningful improvements
- **Model evaluation** should use multiple metrics, not just accuracy
- **Data leakage** prevention is crucial (preprocessing before splitting)

## Next Steps

- Deploy model to production (Flask API, cloud service)
- Monitor model performance over time
- Collect more data for continuous improvement
- Try ensemble methods or deep learning approaches

## Exercises

1. **Data Leakage**: What would happen if we scaled features before splitting? Why?

2. **Feature Selection**: Implement feature selection using mutual information or recursive elimination.

3. **Pipeline Automation**: Create a scikit-learn Pipeline that combines preprocessing and modeling.

4. **Model Interpretability**: Add SHAP or LIME explanations for model predictions.

5. **Production Deployment**: Create a simple Flask API to serve the model predictions.

## Solutions

1. **Data Leakage**: Test set information would leak into training. Scaler would use test set statistics, making evaluation overly optimistic.

2. **Feature Selection**: Use SelectKBest with mutual_info_classif, or RFECV with cross-validation.

3. **Pipeline Automation**: Use sklearn.pipeline.Pipeline with StandardScaler and classifier steps.

4. **Model Interpretability**: SHAP values show feature contribution to each prediction.

5. **Production Deployment**: Flask route accepts JSON features, returns prediction probabilities.